In [1]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
docs = PyPDFDirectoryLoader("papers/").load()
print(len(docs))  # one Document per page, with metadata (source, page)

C:\Users\Zenbook\AppData\Local\Temp\ipykernel_14968\829403855.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


141


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
db = Chroma.from_documents(chunks, embeddings, persist_directory="chroma_db")

c:\AI engineering\Quantum papers RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\AI engineering\Quantum papers RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Zenbook\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activat

In [26]:
retriever = db.as_retriever(search_kwargs={"k": 5})
results = retriever.invoke("How to calculate the ground state energy of a quantum system using VQE?")
for r in results:
    print(r.metadata["source"], r.metadata["page"], r.page_content[:200])

papers\2512.22572v2.pdf 0 tion. At present, we find ourselves in the era of Noisy
Intermediate-Scale Quantum (NISQ) [2] devices, mean-
ing that quantum devices are built from a few hundred
qubits and suffer from qubit errors (
papers\2512.22572v2.pdf 0 Variational quantum eigensolver for chemical molecules
Luca Ion and Adam Smith
School of Physics & Astronomy, University of Nottingham, Nottingham, UK
(Dated: May 16, 2023)
Solving multi-particle syst
papers\2610.29779v1.pdf 0 accelerate molecular simulations [4].
Among the most promising approaches for near-term
quantum devices are Variational Quantum Algorithms
(VQAs)[5], particularly the Variational Quantum Eigen-
solver
papers\2607.02620v1.pdf 1 electronic wavefunctions [9], qubits are capable of representing these systems and then approximating the
ground state of the Hamiltonian variationally using a parameterized trial state [10].
In the c
papers\2611.16610v1.pdf 2 2.2 Variational Quantum Eigensolver
The Variational Quantum Eigens

In [23]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

llm = ChatOpenAI(
    model="crow-uncensored:free",
    base_url="https://crowllm.com/v1",
    api_key=os.getenv("CROW_API_KEY"),
    temperature=0,
)

print(llm.invoke("Say hello in one sentence.").content)

UncensoredAI
Hello.


In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """Answer using ONLY the context below. Cite sources as [source, page].
If the answer is not in the context, say "Not found in the sources."

Context:
{context}

Question: {question}"""
)

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata['source']}, p.{d.metadata['page']}] {d.page_content}" for d in docs
    )

chain = (
    {"context": retriever | format_docs, "question": lambda x: x}
    | prompt | llm | StrOutputParser()
)

print(chain.invoke("Can you explain the mathematical intuition behind a hamiltonian ?"))

UncensoredAI

text
Not found in the sources.


In [25]:
print(chain.invoke("From where does a hamiltonin come from in quantum mechanics?"))

UncensoredAI
Not found in the sources.
